# Integrated System — Gated Scoring + Adaptive Blended Threshold + Incremental Learning

**Purpose**: the gated combined-score approach (`cc1_test` F1 0.618 -> 0.721)
was validated using simple static thresholds per mode. This notebook checks
whether it composes correctly with the two mechanisms already validated
separately in this pipeline — `adaptive_threshold_blended.ipynb` (per-container
shrinkage threshold) and `incremental_learning.ipynb` (KS-test-triggered
SGD fine-tuning) — rather than assuming they just combine for free.

**Two real integration problems solved here, not assumed away:**

1. **The blended threshold's local/global statistics were calibrated against
   VAE-MSE's scale.** The gated scorer's "combined" mode uses a different
   score (max of two z-scores) with a different distribution entirely. This
   notebook maintains SEPARATE per-container history buffers and SEPARATE
   global reference statistics for each mode, so switching modes never mixes
   incompatible scores into one running average.
2. **Two mechanisms react to the same drift signal with different actions.**
   The composition used here: when the periodic KS-test signals drift,
   BOTH switch to VAE-alone scoring (drift-robust) AND begin fine-tuning the
   VAE (adapt it toward the new normal) — coherent, not redundant, since
   fine-tuning takes several cycles to have an effect, during which the
   VAE-alone score is the safer one to trust anyway.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pickle, joblib, os, copy
from collections import deque
from scipy import stats
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, roc_auc_score

BASE      = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
DATA_DIR  = os.path.join(BASE, 'data', 'processed')
WIN_DIR   = os.path.join(BASE, 'data', 'processed', 'windows_cc1')
MODEL_DIR = os.path.join(BASE, 'models')
OUT_DIR   = os.path.join(BASE, 'experiments')
WINDOW_SIZE = 30

FEATURE_COLS = [
    'container_cpu_usage_seconds_rate', 'container_cpu_system_seconds_rate', 'container_cpu_user_seconds_rate',
    'container_memory_usage_bytes', 'container_memory_working_set_bytes', 'container_memory_rss', 'container_memory_cache',
]

class VAE(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Linear(hidden1, hidden2), nn.ReLU())
        self.fc_mu = nn.Linear(hidden2, latent_dim)
        self.fc_lv = nn.Linear(hidden2, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden2), nn.ReLU(), nn.Linear(hidden2, hidden1), nn.ReLU(), nn.Linear(hidden1, input_dim))
    def encode(self, x):
        h = self.encoder(x); return self.fc_mu(h), torch.clamp(self.fc_lv(h), -10, 10)
    def decode(self, z): return self.decoder(z)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar); return mu + torch.randn_like(std) * std
    def forward(self, x):
        mu, logvar = self.encode(x); z = self.reparameterize(mu, logvar); return self.decode(z), mu, logvar
    @torch.no_grad()
    def anomaly_score(self, x):
        self.eval(); mu, _ = self.encode(x); return ((self.decode(mu) - x) ** 2).mean(dim=1)

meta = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_meta.pkl'), 'rb'))
base_model = VAE(meta['input_dim'], meta['hidden1'], meta['hidden2'], meta['latent_dim'])
base_model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'vae_cc1.pt'), map_location='cpu'))
base_model.eval()
CLIP = meta['clip']
pca_bundle = joblib.load(os.path.join(MODEL_DIR, 'cc1_pca.pkl'))
pca = pca_bundle['pca']

eval_results = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_eval.pkl'), 'rb'))
print('Model + PCA loaded.')

Model + PCA loaded.


## Step 1 — Global reference statistics (VAE-alone and combined score), from `cc1_train`/`cc1_val`

In [2]:
X_cc1_train = np.clip(np.load(os.path.join(WIN_DIR, 'X_cc1_train.npy')), -CLIP, CLIP).astype(np.float32)
X_val = np.clip(np.load(os.path.join(WIN_DIR, 'X_cc1_val.npy')), -CLIP, CLIP).astype(np.float32)
y_val = np.load(os.path.join(WIN_DIR, 'y_cc1_val.npy'))

with torch.no_grad():
    vae_mse_train = base_model.anomaly_score(torch.from_numpy(X_cc1_train)).numpy()
    vae_mse_val = base_model.anomaly_score(torch.from_numpy(X_val)).numpy()
gau_train = (X_cc1_train ** 2).sum(axis=1)
gau_val = (X_val ** 2).sum(axis=1)

VAE_MU, VAE_SD = float(vae_mse_train.mean()), float(vae_mse_train.std())
GAU_MU, GAU_SD = float(gau_train.mean()), float(gau_train.std())

def z_vae(a): return (a - VAE_MU) / VAE_SD
def z_gau(a): return (a - GAU_MU) / GAU_SD

comb_train = np.maximum(z_vae(vae_mse_train), z_gau(gau_train))
comb_val = np.maximum(z_vae(vae_mse_val), z_gau(gau_val))
COMB_MU, COMB_SD = float(comb_train.mean()), float(comb_train.std())

print(f'VAE-alone global:  mean={VAE_MU:.5f}  std={VAE_SD:.5f}')
print(f'Combined global:   mean={COMB_MU:.4f}  std={COMB_SD:.4f}')

VAE-alone global:  mean=0.02218  std=0.03201
Combined global:   mean=0.3205  std=1.2260


## Step 2 — Calibrate PRIOR_STRENGTH for EACH mode's blended threshold separately on `cc1_val`

`adaptive_threshold_blended.ipynb` already validated `PRIOR_STRENGTH=500` for
the VAE-alone score (reused here, unchanged). The combined score needs its
own calibration since it has a different distribution — same procedure,
same target (~1% FPR on `cc1_val`, which has zero anomalies).

In [3]:
K_ADAPTIVE = 3.0
BUFFER_SIZE = 500
VAE_PRIOR_STRENGTH = 500   # reused, already validated in adaptive_threshold_blended.ipynb

def run_blended_calibration(scores, global_mean, global_std, prior_strength):
    # cc1_val has one 'container' worth of behavior per row already grouped; treat as one pooled stream
    # for calibration purposes (same simplification as checking pooled FPR, matching the original
    # calibration's own use of cc1_val as an undifferentiated normal stream).
    buf = deque(maxlen=BUFFER_SIZE)
    n_flagged = 0
    for s in scores:
        n_local = len(buf)
        w = n_local / (n_local + prior_strength)
        if n_local == 0:
            lm, ls = global_mean, global_std
        else:
            arr = np.fromiter(buf, dtype=np.float64); lm, ls = arr.mean(), arr.std()
        t = (w * lm + (1 - w) * global_mean) + K_ADAPTIVE * (w * ls + (1 - w) * global_std)
        is_anom = s > t
        n_flagged += int(is_anom)
        if not is_anom:
            buf.append(s)
    return n_flagged / len(scores)

PRIOR_CANDIDATES = [100, 500, 2000, 10000, 50000]
print('Calibrating combined-score PRIOR_STRENGTH on cc1_val:')
comb_fpr = {}
for ps in PRIOR_CANDIDATES:
    fpr = run_blended_calibration(comb_val, COMB_MU, COMB_SD, ps)
    comb_fpr[ps] = fpr
    print(f'  prior_strength={ps:6d}  FPR={fpr*100:.2f}%')

COMB_PRIOR_STRENGTH = min(PRIOR_CANDIDATES, key=lambda ps: abs(comb_fpr[ps] - 0.01))
print(f'\nChosen combined-mode PRIOR_STRENGTH = {COMB_PRIOR_STRENGTH}')

Calibrating combined-score PRIOR_STRENGTH on cc1_val:
  prior_strength=   100  FPR=11.97%
  prior_strength=   500  FPR=1.07%
  prior_strength=  2000  FPR=0.72%
  prior_strength= 10000  FPR=0.61%
  prior_strength= 50000  FPR=0.59%

Chosen combined-mode PRIOR_STRENGTH = 500


## Step 3 — Build true chronological streams (rebuilt from raw CSVs, verified against saved arrays)

In [4]:
def build_stream(csv_name):
    d = pd.read_csv(os.path.join(DATA_DIR, csv_name), low_memory=False)
    d['is_gap'] = d['is_gap'].astype(bool)
    X_raw, cmdb_ids, end_ts, ys = [], [], [], []
    for cmdb_id, g in d.sort_values('timestamp').groupby('cmdb_id'):
        data_arr = g[FEATURE_COLS].values.astype(np.float32)
        is_gap = g['is_gap'].values
        labels = g['label'].values
        ts = g['timestamp'].values
        n = len(g)
        for i in range(0, n - WINDOW_SIZE + 1, 1):
            if is_gap[i:i+WINDOW_SIZE].any():
                continue
            X_raw.append(data_arr[i:i+WINDOW_SIZE])
            cmdb_ids.append(cmdb_id)
            end_ts.append(ts[i+WINDOW_SIZE-1])
            ys.append(int(labels[i:i+WINDOW_SIZE].any()))
    X_raw = np.stack(X_raw)
    X_flat = X_raw.reshape(len(X_raw), -1)
    X_pca = np.clip(pca.transform(X_flat), -CLIP, CLIP).astype(np.float32)
    order = np.argsort(np.array(end_ts), kind='stable')
    return {'X': X_pca[order], 'y': np.array(ys, dtype=np.int64)[order], 'cmdb': np.array(cmdb_ids)[order]}

stream_cc1_test = build_stream('cc1_test.csv')
stream_drift_cc2 = build_stream('drift_complex_case2.csv')
print(f'cc1_test:  {len(stream_cc1_test["y"]):,} windows')
print(f'drift_cc2: {len(stream_drift_cc2["y"]):,} windows')

y_saved_test = np.load(os.path.join(WIN_DIR, 'y_cc1_test.npy'))
y_saved_cc2 = np.load(os.path.join(WIN_DIR, 'y_drift_cc2.npy'))
assert stream_cc1_test['y'].sum() == y_saved_test.sum()
assert stream_drift_cc2['y'].sum() == y_saved_cc2.sum()
print('Anomaly-count sanity checks passed.')

rng = np.random.default_rng(42)
ref_idx = rng.choice(len(X_cc1_train), size=5000, replace=False)
REF_SAMPLE = X_cc1_train[ref_idx]
with torch.no_grad():
    REFERENCE_MSE = base_model.anomaly_score(torch.from_numpy(REF_SAMPLE)).numpy()
print(f'Frozen KS-test reference: {len(REFERENCE_MSE):,} cc1_train windows (scored by the ORIGINAL, never fine-tuned model).')

cc1_test:  44,185 windows
drift_cc2: 76,977 windows
Anomaly-count sanity checks passed.
Frozen KS-test reference: 5,000 cc1_train windows (scored by the ORIGINAL, never fine-tuned model).


## Step 4 — The integrated streaming simulator

Composition rule at each periodic check: **drift detected -> switch to
VAE-alone scoring AND fine-tune the VAE. No drift -> use combined scoring,
no fine-tune.** Per-mode history buffers are kept completely separate so a
mode switch never blends incompatible score scales into one running average.

In [5]:
REFIT_INTERVAL, FT_BUFFER_SIZE, FT_LR, FT_EPOCHS, KS_ALPHA = 5000, 2000, 1e-4, 5, 0.001

def run_integrated_stream(X, cmdb_arr, verbose_name):
    model = copy.deepcopy(base_model)
    opt = torch.optim.Adam(model.parameters(), lr=FT_LR)
    n = len(X)

    mode = 'combined'
    buffers_vae, buffers_comb = {}, {}
    ft_pool = deque(maxlen=FT_BUFFER_SIZE)
    preds = np.zeros(n, dtype=np.int64)
    switch_events, finetune_events = [], []

    for i in range(n):
        x_t = torch.from_numpy(X[i:i+1])
        with torch.no_grad():
            mse = model.anomaly_score(x_t).item()
        gau = float((X[i] ** 2).sum())
        combined = max((mse - VAE_MU) / VAE_SD, (gau - GAU_MU) / GAU_SD)
        cid = cmdb_arr[i]

        if mode == 'combined':
            score, global_mean, global_std, prior_strength = combined, COMB_MU, COMB_SD, COMB_PRIOR_STRENGTH
            buf = buffers_comb.setdefault(cid, deque(maxlen=BUFFER_SIZE))
        else:
            score, global_mean, global_std, prior_strength = mse, VAE_MU, VAE_SD, VAE_PRIOR_STRENGTH
            buf = buffers_vae.setdefault(cid, deque(maxlen=BUFFER_SIZE))

        n_local = len(buf)
        w = n_local / (n_local + prior_strength)
        if n_local == 0:
            lm, ls = global_mean, global_std
        else:
            arr = np.fromiter(buf, dtype=np.float64); lm, ls = arr.mean(), arr.std()
        threshold = (w * lm + (1 - w) * global_mean) + K_ADAPTIVE * (w * ls + (1 - w) * global_std)

        is_anom = score > threshold
        preds[i] = int(is_anom)
        if not is_anom:
            buf.append(score)
            ft_pool.append(X[i])

        if (i + 1) % REFIT_INTERVAL == 0 and len(ft_pool) >= FT_BUFFER_SIZE // 2:
            pooled = np.stack(list(ft_pool))
            with torch.no_grad():
                recent_mse = model.anomaly_score(torch.from_numpy(pooled)).numpy()
            _, p_value = stats.ks_2samp(REFERENCE_MSE, recent_mse)
            new_mode = 'vae_alone' if p_value < KS_ALPHA else 'combined'
            if new_mode != mode:
                switch_events.append((i + 1, mode, new_mode, round(float(p_value), 8)))
            mode = new_mode

            if new_mode == 'vae_alone':
                Xb = torch.from_numpy(pooled)
                model.train()
                for _ in range(FT_EPOCHS):
                    opt.zero_grad()
                    recon, mu, logvar = model(Xb)
                    recon_loss = nn.functional.mse_loss(recon, Xb, reduction='mean')
                    kl = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
                    (recon_loss + 0.01 * kl).backward()
                    opt.step()
                model.eval()
                finetune_events.append(i + 1)

    return preds, switch_events, finetune_events

print('Integrated streaming simulator defined.')

Integrated streaming simulator defined.


In [6]:
results_integrated = {}
for name, stream in [('cc1_test', stream_cc1_test), ('drift_cc2', stream_drift_cc2)]:
    preds, switches, finetunes = run_integrated_stream(stream['X'], stream['cmdb'], name)
    y = stream['y']
    p = precision_score(y, preds, zero_division=0)
    r = recall_score(y, preds, zero_division=0)
    f1 = f1_score(y, preds, zero_division=0)
    results_integrated[name] = {'precision': p, 'recall': r, 'f1': f1, 'switch_events': switches, 'finetune_events': finetunes}
    print(f'=== {name} ===')
    print(f'  mode switches: {switches if switches else "(none)"}')
    print(f'  fine-tune events: {len(finetunes)}  {finetunes if finetunes else ""}')
    print(f'  precision={p:.3f}  recall={r:.3f}  F1={f1:.3f}\n')

=== cc1_test ===
  mode switches: [(5000, 'combined', 'vae_alone', 2.368e-05), (10000, 'vae_alone', 'combined', 0.0908456), (20000, 'combined', 'vae_alone', 1e-08), (40000, 'vae_alone', 'combined', 0.19640619)]
  fine-tune events: 5  [5000, 20000, 25000, 30000, 35000]
  precision=0.807  recall=0.621  F1=0.702

=== drift_cc2 ===
  mode switches: [(5000, 'combined', 'vae_alone', 0.0)]
  fine-tune events: 15  [5000, 10000, 15000, 20000, 25000, 30000, 35000, 40000, 45000, 50000, 55000, 60000, 65000, 70000, 75000]
  precision=0.251  recall=0.767  F1=0.378



## Step 5 — Full comparison against every previously-tested configuration

In [7]:
print(f'{"set":12s} {"method":32s} {"Precision":>10s} {"Recall":>8s} {"F1":>7s}')
rows = [
    ('cc1_test', 'VAE alone (deployed)', 0.630, 0.605, 0.618),
    ('cc1_test', '+ Adaptive blended threshold', 0.676, 0.578, 0.623),
    ('cc1_test', '+ Incremental learning (Full Model)', 0.701, 0.578, 0.634),
    ('cc1_test', 'Gated scoring alone (static thresh)', 0.838, 0.707, 0.767),
    ('cc1_test', 'GATED + blended threshold', 0.838, 0.707, 0.767),
    ('drift_cc2', 'VAE alone (deployed)', 0.168, 0.772, 0.276),
    ('drift_cc2', '+ Adaptive blended threshold', 0.223, 0.786, 0.347),
    ('drift_cc2', '+ Incremental learning (Full Model)', 0.262, 0.768, 0.391),
    ('drift_cc2', 'Gated scoring alone (static thresh)', 0.110, 0.865, 0.195),
    ('drift_cc2', 'GATED + blended threshold', 0.164, 0.772, 0.271),
]
for name, method, p, r, f1 in rows:
    print(f'{name:12s} {method:32s} {p:10.3f} {r:8.3f} {f1:7.3f}')
print()
for name in ['cc1_test', 'drift_cc2']:
    r = results_integrated[name]
    print(f'{name:12s} {"INTEGRATED (gated+blended+IL)":32s} {r["precision"]:10.3f} {r["recall"]:8.3f} {r["f1"]:7.3f}')

set          method                            Precision   Recall      F1
cc1_test     VAE alone (deployed)                  0.630    0.605   0.618
cc1_test     + Adaptive blended threshold          0.676    0.578   0.623
cc1_test     + Incremental learning (Full Model)      0.701    0.578   0.634
cc1_test     Gated scoring alone (static thresh)      0.838    0.707   0.767
cc1_test     GATED + blended threshold             0.838    0.707   0.767
drift_cc2    VAE alone (deployed)                  0.168    0.772   0.276
drift_cc2    + Adaptive blended threshold          0.223    0.786   0.347
drift_cc2    + Incremental learning (Full Model)      0.262    0.768   0.391
drift_cc2    Gated scoring alone (static thresh)      0.110    0.865   0.195
drift_cc2    GATED + blended threshold             0.164    0.772   0.271

cc1_test     INTEGRATED (gated+blended+IL)         0.807    0.621   0.702
drift_cc2    INTEGRATED (gated+blended+IL)         0.251    0.767   0.378


## Step 6 — Save (does NOT overwrite the deployed model or existing gated checkpoint)

In [8]:
save_results = {
    'vae_mu': VAE_MU, 'vae_sd': VAE_SD, 'gau_mu': GAU_MU, 'gau_sd': GAU_SD,
    'comb_mu': COMB_MU, 'comb_sd': COMB_SD,
    'vae_prior_strength': VAE_PRIOR_STRENGTH, 'comb_prior_strength': COMB_PRIOR_STRENGTH,
    'k_adaptive': K_ADAPTIVE, 'buffer_size': BUFFER_SIZE,
    'gate_config': {'refit_interval': REFIT_INTERVAL, 'ft_buffer_size': FT_BUFFER_SIZE, 'ft_lr': FT_LR, 'ft_epochs': FT_EPOCHS, 'ks_alpha': KS_ALPHA},
    'results': results_integrated,
}
out_path = os.path.join(OUT_DIR, 'integrated_system_results.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(save_results, f)
print(f'Saved -> {out_path}')

Saved -> c:\Users\jthar\Documents\Claude\Projects\module3\module3\experiments\integrated_system_results.pkl


## How to read this

- **Step 2** solves integration problem #1 (recalibrating the blended
  threshold's global reference for the combined score's own scale, rather
  than reusing VAE-MSE-tuned constants blindly).
- **Step 4** solves integration problem #2 (a specific, coherent composition
  rule for what happens when drift is detected — switch scoring mode AND
  fine-tune — rather than assuming the two mechanisms don't interfere).
- **Step 5 is the honest verdict**: does layering all three mechanisms
  together beat each one alone? If the integrated row matches or beats the
  best of the individual rows on both sets, the composition works. If it's
  worse than the gated-alone row on `cc1_test`, that means the added
  adaptive-threshold/fine-tuning layers cost something there — a real,
  reportable trade-off, not assumed away.